In [ ]:
#logistc regression
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
recipes = pd.read_json("train.json")  
recipes.head()

In [ ]:
print(recipes.shape)
print(recipes.columns)
print(recipes["cuisine"].nunique(), "cuisines")

In [ ]:
recipes["ingredients_text"] = recipes["ingredients"].apply(lambda xs: " ".join(xs).lower())
X = recipes["ingredients_text"]
y = recipes["cuisine"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),   # helps a lot
    min_df=2
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(X_train_vec.shape, X_test_vec.shape)

In [ ]:
model = LogisticRegression(
    max_iter=2000
)

start = time.time()
model.fit(X_train_vec, y_train)
train_time = time.time() - start

start = time.time()
pred = model.predict(X_test_vec)
pred_time = time.time() - start

print("Training time (s):", round(train_time, 3))
print("Prediction time (s):", round(pred_time, 3))

In [ ]:
acc = accuracy_score(y_test, pred)
print("Accuracy:", round(acc, 4))
print(classification_report(y_test, pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Logistic Regression Confusion Matrix')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

report = classification_report(y_test, pred, output_dict=True)
cuisines = list(model.classes_)
f1_scores = [report[c]['f1-score'] for c in cuisines]

plt.figure(figsize=(10, 8))
plt.barh(cuisines, f1_scores)
plt.xlabel('F1 Score')
plt.title('Logistic Regression — F1 Score per Cuisine')
plt.tight_layout()
plt.show()

In [ ]:
def predict_cuisine(ingredients):
    
    text = " ".join(ingredients).lower()
    
    vec = tfidf.transform([text])
    
    prediction = model.predict(vec)
    
    return prediction[0]

In [ ]:
predict_cuisine(["rice","salmon"])